# Fock ML Descriptors

This tutorial demonstrates how to extract the Fock Matrix from an ORCA calculation using the ORCA python interfacte (OPI). After extraction the Fock matrix is transformed into the Boys-localized internal orbital and localized intrinsic valence virtual orbital (LIVVO) spaces.

In this notebook we will:
1. Import the Required Dependencies
2. Define a Working Directory and Prepare Structures
3. Download and Prepare Molecular Structure from PubChem
4. Run ORCA Calculations
5. Extract Matrices and Perform Fock Matrix Diagonalization

## Step 1: Import Dependencies

We begin by importing all required Python modules for this tutorial.

In [ ]:
import json
import shutil

import numpy as np
from pathlib import Path
from scipy.io import loadmat

import subprocess
from pathlib import Path
from opi.core import Calculator
from opi.input.structures.structure import Structure
from opi.input.simple_keywords import Scf, Method, BasisSet

float_formatter = "{:.5f}".format
np.set_printoptions(formatter={'float_kind':float_formatter})

In [12]:
def utri2mat(utri: np.ndarray) -> np.ndarray:
    """Convert rolled-out upper triangle to full matrix"""
    n = int(-1 + np.sqrt(1 + 8*len(utri))) // 2
    iu1 = np.triu_indices(n)
    ret = np.empty((n, n))
    ret[iu1] = utri
    ret.T[iu1] = utri
    return ret

def trim_array(arr: np.ndarray) -> np.ndarray:
    """Trim a 1-D array to the last non-zero element"""
    if arr.ndim != 1:
        raise ValueError("Array must be 1-dimensional")
    return arr[:np.nonzero(arr)[0][-1]+1]

In [13]:
def read_orbnet(file: str | Path) -> tuple[int, np.ndarray]:
    """Extract some features from an OrbNet data file

    Returns
    -------
    nocc
        Number of valence occupied MOs
    f_diag
        Diagonal Fock matrix elements, occ + virt, each separately sorted in ascending order
    """
    diag = loadmat(str(file))['features_diag_ccpVTZ']
    nocc = diag.shape[0]
    f_occ_diag = diag[:,0]
    f_virt = utri2mat(diag[0,1:211])
    f_virt_diag = trim_array(f_virt.diagonal())
    f_diag = np.concatenate((np.sort(f_occ_diag), np.sort(f_virt_diag)))
    return nocc, f_diag

In [14]:
# read the orbnet data
orbnet_files = sorted(Path('orbnet_data').glob('*.mat'))
molnames = [p.stem.rsplit('_', 1)[0] for p in orbnet_files]
orbnet_data = {mol: read_orbnet(file) for mol, file in zip(molnames, orbnet_files)}

In [15]:
# ORCA parsing/processing functions
def orca_mos(orca_json: dict) -> np.ndarray:
    """Helper function to get MO coefficients from the ORCA JSON"""
    return np.column_stack([mo['MOCoefficients'] for mo in orca_json['Molecule']['MolecularOrbitals']['MOs']])

def orca_loc_orbwins(orca_output: Path) -> tuple[tuple[int, int] | None, tuple[int, int] | None]:
    """Read the valence occupied and valence virtual localized orbital windows from the ORCA output file"""
    occ = None
    virt = None
    with open(orca_output) as f:
        for line in f:
            if not occ and 'Orbital range for localization' in line:
                fields = line.split()
                occ = int(fields[-3]), int(fields[-1])
                continue
            if not virt and 'Orbital range for VVO localization' in line:
                fields = line.split()
                virt = int(fields[-3]), int(fields[-1])
                break
    return occ, virt

def orca_process(outfile: Path, jsonfile: Path) -> tuple[int, np.ndarray]:
    """Extract some features from ORCA output and JSON files

    Returns
    -------
    nocc
        Number of valence occupied MOs
    f_diag
        Diagonal Fock matrix elements, occ + virt, each separately sorted in ascending order
    """
    # read orbital windows from the output
    orbwin_occ, orbwin_vvo = orca_loc_orbwins(outfile)
    nocc = orbwin_occ[1] - orbwin_occ[0] + 1
    nvvo = orbwin_vvo[1] - orbwin_vvo[0] + 1
    # read ORCA JSON file
    with open(jsonfile) as f:
        json_data = json.load(f)
    # localized MO coefficients
    c = orca_mos(json_data)
    # trim to valence + VVO
    c_val = c[:,orbwin_occ[0]:orbwin_vvo[1]+1]
    # one-electron Hamiltonian
    h = np.array(json_data['Molecule']['H-Matrix'])
    # two-electron Fock matrix (alpha)
    g = np.array(json_data['Molecule']['F-Matrix'][0])
    # total Fock matrix
    fao = h + g
    # transform to localized MOs
    fmo = c_val.T @ fao @ c_val
    f_diag = fmo.diagonal()
    # sort occ and virt elements
    f_diag = np.concatenate((np.sort(f_diag[:nocc]), np.sort(f_diag[nocc:])))
    return nocc, f_diag

def orca_run(xyz_file: Path) -> None:
    """Run an ORCA quantum chemistry calculation from a given XYZ file.

    Parameters
    ----------
    xyz_file : Path
        Path to the input .xyz file. A directory with the same stem name will be created
        in the same location to store ORCA outputs.
    """
    xyz_file = Path(xyz_file)
    mol_name = xyz_file.stem
    structure = Structure.from_xyz(xyz_file)
    working_dir = xyz_file.parent / mol_name
    working_dir.mkdir(exist_ok=True)
    calc = Calculator(basename=mol_name, working_dir=working_dir)
    calc.structure = structure
    calc.structure.charge = 0
    calc.structure.multiplicity = 1
    calc.add_simple_keywords(
        Scf.TIGHTSCF,
        Scf.NOSLOPPYSCFCHECK,
        Scf.PMODEL,
        BasisSet.CC_PVTZ,
        Method.HF,
    )
    calc.add_arbitrary_string(
        "%loc locmet NEWBOYS locmetvirt LIVVO tol 1e-8 occ true virt true iaobasis minao_auto_pp end",
        top=True
    )
    calc.write_input()
    calc.run()
    source_conf = xyz_file.parent / "orca.json.conf"
    target_conf = working_dir / f"{mol_name}.json.conf"
    shutil.copy(source_conf, target_conf)
    gbw_file = working_dir / f"{mol_name}.loc"
    cmd = ["orca_2json", str(gbw_file)]
    subprocess.run(cmd, check=True)

In [18]:
orca_run(Path("./orca_run/water.xyz"))
orca_run(Path("./orca_run/butane.xyz"))
orca_run(Path("./orca_run/ethane.xyz"))
orca_run(Path("./orca_run/isobutane.xyz"))
orca_run(Path("./orca_run/methane.xyz"))
orca_run(Path("./orca_run/propane.xyz"))

----------
ORCA_2JSON
----------

Reading the gbw file     ... orca_run/water/water.loc done
Creating JSON Objects    ... 
Calculating 1-electron integrals (SHTV)        ... done 
Calculating dipole integrals                   ... done 

-------------------
DFT GRID GENERATION
-------------------

General Integration Accuracy     IntAcc      ... 4.388
Radial Grid Type                 RadialGrid  ... OptM3 with GC (2021)
Angular Grid (max. ang.)         AngularGrid ... 4 (Lebedev-302)
Angular grid pruning method      GridPruning ... 4 (adaptive)
Weight generation scheme         WeightScheme... mBecke (2022)
Basis function cutoff            BFCut       ... 1.0000e-11
Integration weight cutoff        WCut        ... 1.0000e-14
Partially contracted basis set               ... off
Rotationally invariant grid construction     ... off
Angular grids for H and He will be reduced by one unit

Total number of grid points                  ...    12769
Total number of batches                      .

In [22]:
# read the ORCA data
xyz_files = list(Path("orca_run").glob("*.xyz"))
molnames = [f.stem for f in xyz_files]
base_dir = Path("orca_run")
orca_outfiles = [base_dir / mol / f"{mol}.out" for mol in molnames]
orca_jsonfiles = [base_dir / mol / f"{mol}.json" for mol in molnames]
orca_data = {mol: orca_process(outfile, jsonfile) for mol, outfile, jsonfile in zip(molnames, orca_outfiles, orca_jsonfiles)}

In [23]:
# compare
for mol in molnames:
    nocc_orbnet, fdiag_orbnet = orbnet_data[mol]
    nocc_orca, fdiag_orca = orca_data[mol]
    #print(orbnet_data[mol])
    #print(orca_data[mol])
    if nocc_orca != nocc_orbnet:
        print(f'{mol}: {nocc_orca=} != {nocc_orbnet=}')
    elif not np.allclose(fdiag_orca, fdiag_orbnet):
        if not np.allclose(fdiag_orca[:nocc_orca], fdiag_orbnet[:nocc_orbnet]):
            maxdiff = np.max(np.abs(fdiag_orca[:nocc_orca] - fdiag_orbnet[:nocc_orbnet]))
            print(f'{mol}: Focc differs by {maxdiff}!')
        if not np.allclose(fdiag_orca[nocc_orca:], fdiag_orbnet[nocc_orbnet:]):
            maxdiff = np.max(np.abs(fdiag_orca[nocc_orca:] - fdiag_orbnet[nocc_orbnet:]))
            print(f'{mol}: Fvirt differs by {maxdiff}!')
    else:
        print(f'{mol}: OK!')

ethane: Fvirt differs by 2.0596795837435344e-05!
isobutane: Fvirt differs by 2.7732973327254662e-05!
propane: Fvirt differs by 2.2881152004217142e-05!
water: OK!
butane: Fvirt differs by 2.151666882121983e-05!
methane: Fvirt differs by 1.7877504689733925e-05!
